# Documentary GPU — Colab レンダリング
ランタイム → ハードウェアアクセラレータ → **GPU (T4)** を選択。

設計の根拠は `docs/research_2026.md`。主役は2.5Dパララックス。

In [ ]:
# セル1: GitHub から取得（リポジトリ公開後）+ Drive マウント
import os, subprocess
from google.colab import drive
drive.mount('/content/drive')
WORK='/content/documentary_gpu'
# git clone は公開後に。当面は Drive からコピー or 手動アップロード。
print('WORK =', WORK)

In [ ]:
# セル2: 依存インストール（コア + GPU）
!apt-get -q install -y ffmpeg fonts-ipafont fonts-noto-cjk
!pip install -q torch numpy Pillow transformers diffusers accelerate sentencepiece
!pip install -q edge-tts soundfile imageio imageio-ffmpeg
# 任意（重い・必要時に）:
# !pip install -q depthflow style-bert-vits2 bitsandbytes

In [ ]:
# セル3: 環境チェック
%cd $WORK
!python scripts/pipeline.py doctor

In [ ]:
# セル4: アセット取り込み（前プロジェクトの画像/BGMを Drive 経由で）
!mkdir -p assets/images assets/bgm
!cp /content/drive/MyDrive/documentary_gpu_assets/images/* assets/images/ 2>/dev/null || echo 'images: Drive未配置'
!cp /content/drive/MyDrive/documentary_gpu_assets/bgm/* assets/bgm/ 2>/dev/null || echo 'bgm: Drive未配置'
!ls assets/images | head

In [ ]:
# セル5: テストシーンを1本レンダリング（パララックス確認）
!python scripts/pipeline.py render-scene scenes/test.json test_luther
from IPython.display import Video
Video('build/clips/test_luther.mp4', embed=True, width=720)

In [ ]:
# セル6: 章まるごとレンダリング
!python scripts/pipeline.py render test
from IPython.display import Video
Video('build/chapters/test.mp4', embed=True, width=720)

In [ ]:
# セル7: Drive に保存
import shutil, glob, os
OUT='/content/drive/MyDrive/documentary_gpu_output'
os.makedirs(OUT, exist_ok=True)
for f in glob.glob('build/chapters/*.mp4'):
    shutil.copy(f, OUT); print('保存:', os.path.basename(f))